[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/numerical_methods/01_error_analysis_and_floating_point/exercises.ipynb)

# Exercises — Topic 01: Error Analysis and Floating Point

20 fully solved problems in 4 levels: concept checks, foundational error analysis, AI/ML and physics applications, and challenge proofs.

## Level 0 — Concept Check

### Problem L0.1: Absolute vs relative error

Approximate $\pi \approx 3.14159265$ by $\hat{x} = 3.14$. Compute the absolute and relative errors, and state how many significant decimal digits $\hat{x}$ carries.

**Solution.**

Absolute error:

$$
E_{\mathrm{abs}} = \lvert 3.14 - 3.14159265 \rvert = 1.59265 \times 10^{-3}.
$$

Relative error:

$$
E_{\mathrm{rel}} = \frac{1.59265 \times 10^{-3}}{3.14159265} \approx 5.07 \times 10^{-4}.
$$

Since $5.07 \times 10^{-4} \le \tfrac{1}{2} \times 10^{-3+1} \times 10^{-1}$ fails but $5.07 \times 10^{-4} \le \tfrac{1}{2} \times 10^{-2}$ holds, we check Definition 2: $E_{\mathrm{rel}} \le \tfrac{1}{2}\times 10^{-p+1}$ gives $p = 3$.

$$
\boxed{E_{\mathrm{abs}} \approx 1.59 \times 10^{-3}, \quad E_{\mathrm{rel}} \approx 5.07 \times 10^{-4}, \quad p = 3 \text{ significant digits}}
$$

*Key takeaway:* Relative error, not absolute error, determines the count of significant digits.

### Problem L0.2: Machine epsilon

In IEEE double precision, what is the result of the expression $\mathrm{fl}(1 + 10^{-20})$, and why? What is the smallest $x \gt 0$ for which $\mathrm{fl}(1 + x) \gt 1$?

**Solution.**

Representable numbers near $1$ are spaced $\varepsilon_{\mathrm{mach}} = 2^{-52} \approx 2.22 \times 10^{-16}$ apart. Since $10^{-20}$ is far smaller than half this spacing, the exact value $1 + 10^{-20}$ rounds back to $1$:

$$
\mathrm{fl}(1 + 10^{-20}) = 1.
$$

With round-to-nearest-even, $\mathrm{fl}(1 + x) \gt 1$ first occurs when $x$ exceeds half the gap, i.e. just above the unit roundoff $u = 2^{-53}$ (values of $x$ slightly greater than $2^{-53}$ round up to $1 + 2^{-52}$).

$$
\boxed{\mathrm{fl}(1 + 10^{-20}) = 1; \quad \text{threshold} \approx u = 2^{-53} \approx 1.11 \times 10^{-16}}
$$

*Key takeaway:* Numbers are absorbed in addition when they fall below half the local spacing of the floating-point grid.

### Problem L0.3: Truncation vs rounding error

Classify the error source in each computation: (a) approximating $e = \sum_{k=0}^{\infty} 1/k!$ by the first 10 terms in exact rational arithmetic; (b) evaluating $\sin(0.5)$ with the standard library in double precision; (c) approximating $f'(x)$ by $(f(x+h) - f(x))/h$ in double precision.

**Solution.**

(a) The arithmetic is exact; the only error is from cutting off the infinite series — pure **truncation error**, of size $\sum_{k \ge 10} 1/k! \lt 2/10! \approx 5.5 \times 10^{-7}$.

(b) The library evaluates a polynomial/table approximation accurate to well below $u$; the returned value is the correctly rounded $\sin(0.5)$ — the error is essentially pure **rounding error**, bounded by $u \lvert \sin(0.5) \rvert$.

(c) Both sources are present: replacing the limit by a finite $h$ contributes truncation error $O(h)$, while the cancellation in $f(x+h) - f(x)$ contributes rounding error $O(u/h)$.

$$
\boxed{\text{(a) truncation only; (b) rounding only; (c) both, } O(h) + O(u/h)}
$$

*Key takeaway:* Truncation error is a property of the mathematical approximation; rounding error is a property of finite-precision arithmetic — they must be budgeted separately.

### Problem L0.4: Why is subtraction the dangerous operation?

Addition of positive numbers, multiplication and division are all well-conditioned. Explain in one line of algebra why subtraction of nearly equal numbers is the sole ill-conditioned basic operation.

**Solution.**

Condition number of $f(a, b) = a - b$ with respect to relative perturbations of the data (Proof 2 of the theory notebook):

$$
\kappa_{-}(a, b) = \frac{\lvert a \rvert + \lvert b \rvert}{\lvert a - b \rvert} \xrightarrow[\; b \to a \;]{} \infty .
$$

For multiplication, $\widehat{ab} = ab(1+\varepsilon_a)(1+\varepsilon_b) \approx ab(1 + \varepsilon_a + \varepsilon_b)$, so $\kappa_{\times} \le 2$; similarly division, and addition of same-signed numbers has $\kappa_{+} = 1$. Only when $a - b$ is much smaller than $a$ and $b$ do relative input errors get magnified.

$$
\boxed{\kappa_{-} = \dfrac{\lvert a \rvert + \lvert b \rvert}{\lvert a - b \rvert} \to \infty \text{ as } b \to a}
$$

*Key takeaway:* Cancellation is a conditioning problem of the data, not a rounding problem of the operation — the subtraction itself is exact when operands are close (Sterbenz).

## Level 1 — Foundation

### Problem L1.1: The standard model in action

Using the standard model, show that for $a, b, c \gt 0$ the computed value of $s = a + b + c$ (evaluated left to right) satisfies $\hat{s} = a(1+\theta_a) + b(1+\theta_b) + c(1+\theta_c)$ with $\lvert \theta_a \rvert, \lvert \theta_b \rvert \le 2u + u^2$ and $\lvert \theta_c \rvert \le u$. Interpret this as a backward error result.

**Solution.**

Two rounded additions:

$$
\hat{s} = \bigl( (a + b)(1 + \delta_1) + c \bigr)(1 + \delta_2), \qquad \lvert \delta_1 \rvert, \lvert \delta_2 \rvert \le u .
$$

Expanding,

$$
\hat{s} = a(1+\delta_1)(1+\delta_2) + b(1+\delta_1)(1+\delta_2) + c(1+\delta_2).
$$

Set $1 + \theta_a = 1 + \theta_b = (1+\delta_1)(1+\delta_2)$ and $1 + \theta_c = 1 + \delta_2$. Then

$$
\lvert \theta_a \rvert = \lvert \delta_1 + \delta_2 + \delta_1\delta_2 \rvert \le 2u + u^2, \qquad \lvert \theta_c \rvert \le u .
$$

**Backward interpretation:** the computed sum is the *exact* sum of data perturbed relatively by at most $\approx 2u$ — recursive summation is backward stable.

$$
\boxed{\hat{s} = \sum \text{(inputs each perturbed by at most } \approx 2u \text{)}}
$$

*Key takeaway:* Backward error analysis pushes rounding errors onto the data, where their effect is judged by the problem's conditioning.

### Problem L1.2: Catastrophic cancellation, concretely

Evaluate $f(x) = 1 - \cos x$ at $x = 10^{-8}$ (i) naively in double precision, (ii) via the identity $1 - \cos x = 2\sin^2(x/2)$. Estimate the relative error of each.

**Solution.**

The exact value is $1 - \cos(10^{-8}) \approx x^2/2 = 5 \times 10^{-17}$.

(i) **Naive:** $\cos(10^{-8})$ rounds to the double nearest $1 - 5\times 10^{-17}$; but the spacing of doubles near $1$ is $2.22 \times 10^{-16}$, larger than $5 \times 10^{-17}$, so $\mathrm{fl}(\cos 10^{-8}) = 1$ and the computed difference is $0$ — **100% relative error**.

(ii) **Rewritten:** $\sin(x/2) \approx 5 \times 10^{-9}$ is computed with relative error $\approx u$; squaring and doubling gives relative error $\approx 3u$:

$$
2\sin^2(x/2) = 5 \times 10^{-17}(1 + O(u)).
$$

$$
\boxed{\text{naive: } 0 \text{ (total loss); rewrite: } 5 \times 10^{-17} \text{ with relative error } O(u)}
$$

*Key takeaway:* An algebraically equivalent form can move a computation from 0 correct digits to 16 — equivalence in $\mathbb{R}$ is not equivalence in $F$.

### Problem L1.3: Stable quadratic formula

Solve $x^2 - 10^{8} x + 1 = 0$ in double precision. Show that the textbook formula destroys the small root and derive the stable alternative.

**Solution.**

Here $b = -10^8$, and $\sqrt{b^2 - 4ac} = \sqrt{10^{16} - 4} = 10^{8}\sqrt{1 - 4 \times 10^{-16}} \approx 10^{8} - 2\times 10^{-8}$.

The large root is safe: $x_1 = \tfrac{1}{2}\bigl(10^8 + \sqrt{10^{16} - 4}\bigr) \approx 10^{8}$.

The small root via the textbook formula subtracts two numbers agreeing to 16 digits:

$$
x_2 = \frac{10^{8} - \sqrt{10^{16} - 4}}{2},
$$

where the numerator's true value $\approx 2 \times 10^{-8}$ is smaller than the spacing of doubles near $10^{8}$ (which is $\approx 1.5 \times 10^{-8}$) — the computed difference has essentially no correct digits. Instead use Vieta: $x_1 x_2 = c/a = 1$, so

$$
x_2 = \frac{1}{x_1} = \frac{2}{10^{8} + \sqrt{10^{16} - 4}} \approx 1.0000000000000000 \times 10^{-8}.
$$

$$
\boxed{x_1 \approx 10^{8}, \qquad x_2 = \frac{c}{a x_1} \approx 10^{-8} \text{ (stable via Vieta)}}
$$

*Key takeaway:* Compute the large root with the sign that avoids cancellation, then recover the small root from the product of roots.

### Problem L1.4: Condition number of elementary functions

Compute the relative condition number $\kappa_f(x) = \lvert x f'(x) / f(x) \rvert$ for (a) $f(x) = e^{x}$, (b) $f(x) = \ln x$, (c) $f(x) = x^{n}$. Identify where each is ill-conditioned.

**Solution.**

(a) $f' = e^{x}$, so

$$
\kappa(x) = \left\lvert \frac{x e^{x}}{e^{x}} \right\rvert = \lvert x \rvert :
$$

exponentials are ill-conditioned only for large $\lvert x \rvert$ (each unit of $x$ multiplies the output error).

(b) $f' = 1/x$, so

$$
\kappa(x) = \left\lvert \frac{x \cdot (1/x)}{\ln x} \right\rvert = \frac{1}{\lvert \ln x \rvert} \to \infty \text{ as } x \to 1 :
$$

logarithms are ill-conditioned near $x = 1$ (this is exactly why `log1p` exists).

(c) $f' = n x^{n-1}$, so

$$
\kappa(x) = \left\lvert \frac{x \cdot n x^{n-1}}{x^{n}} \right\rvert = n .
$$

$$
\boxed{\kappa_{e^x} = \lvert x \rvert, \qquad \kappa_{\ln} = \frac{1}{\lvert \ln x \rvert}, \qquad \kappa_{x^n} = n}
$$

*Key takeaway:* A power $x^n$ multiplies relative error by exactly $n$; a log near 1 can multiply it without bound.

### Problem L1.5: How many digits survive?

A linear system solver is backward stable and is applied to a matrix with condition number $\kappa(A) = 10^{9}$ in double precision. Estimate the relative error of the computed solution and the number of trustworthy decimal digits.

**Solution.**

Backward stability delivers the exact solution of a system perturbed relatively by $c\,u \approx 10^{-16}$ (modest constant $c$). Conditioning amplifies this by at most $\kappa(A)$:

$$
\frac{\lVert \hat{x} - x \rVert}{\lVert x \rVert} \lesssim \kappa(A)\, c\, u \approx 10^{9} \times 10^{-16} = 10^{-7}.
$$

Trustworthy digits:

$$
\text{digits} \approx 16 - \log_{10} \kappa(A) = 16 - 9 = 7 .
$$

$$
\boxed{\text{relative error} \approx 10^{-7}, \text{ about 7 reliable decimal digits}}
$$

*Key takeaway:* The heuristic "digits lost $\approx \log_{10}\kappa$" converts a condition number into an accuracy budget instantly.

### Problem L1.6: Ordering a sum

You must compute $S = 1 + \sum_{k=1}^{10^{6}} 10^{-16}$ in double precision. Compare the result of summing left-to-right (starting from 1) with summing the small terms first. Explain using absorption.

**Solution.**

**Left-to-right:** the running sum is $1$ after the first term; every subsequent addition computes $\mathrm{fl}(1 + 10^{-16})$. Since $10^{-16} \lt u = 1.11 \times 10^{-16}$, each small term is absorbed:

$$
\hat{S}_{\mathrm{ltr}} = 1 .
$$

**Small terms first:** $\sum_{k=1}^{10^6} 10^{-16} = 10^{-10}$ accumulates essentially exactly (partial sums stay near $10^{-10}$, where the grid spacing is $\approx 10^{-26}$), and the final addition gives

$$
\hat{S}_{\mathrm{sf}} = \mathrm{fl}(1 + 10^{-10}) = 1 + 10^{-10} \text{ (representable to } u).
$$

The true sum is $1 + 10^{-10}$, so left-to-right loses the entire contribution of a million terms.

$$
\boxed{\hat{S}_{\mathrm{ltr}} = 1 \text{ (wrong)}, \qquad \hat{S}_{\mathrm{sf}} = 1 + 10^{-10} \text{ (correct)}}
$$

*Key takeaway:* Sum ascending in magnitude — small terms must be given the chance to coalesce before meeting large ones.

## Level 2 — Applications in AI/ML & Physics

### Problem L2.1: The log-sum-exp trick

Softmax on logits $z = (1000, 1000.5, 999)$ overflows double precision ($e^{1000} \gt 10^{308}$). Prove softmax's shift invariance and compute the correct probabilities.

**Solution.**

For any shift $c$,

$$
\frac{e^{z_i - c}}{\sum_j e^{z_j - c}} = \frac{e^{z_i} e^{-c}}{e^{-c} \sum_j e^{z_j}} = \frac{e^{z_i}}{\sum_j e^{z_j}},
$$

so softmax is invariant under $z \mapsto z - c$. Choose $c = \max_i z_i = 1000.5$; the shifted logits are $(-0.5, 0, -1.5)$, all exponentials lie in $(0, 1]$ — no overflow. Then

$$
e^{-0.5} \approx 0.6065,\quad e^{0} = 1,\quad e^{-1.5} \approx 0.2231,\qquad \Sigma \approx 1.8296,
$$

giving probabilities $\approx (0.3315, 0.5466, 0.1220)$.

$$
\boxed{\mathrm{softmax}(z) \approx (0.331,\; 0.547,\; 0.122) \text{ via the shift } c = \max_i z_i}
$$

*Key takeaway:* Every production softmax/cross-entropy implementation is an error-analysis theorem in disguise: subtract the max, exponentiate, then normalize.

### Problem L2.2: Loss scaling for float16 training

In float16 the smallest positive normal number is $2^{-14} \approx 6.1 \times 10^{-5}$ (subnormals reach $\approx 6.0 \times 10^{-8}$). A network's gradients are concentrated around $10^{-6}$. Explain why unscaled float16 training stalls, and find the smallest power-of-two loss scale that moves the gradients into the normal range.

**Solution.**

Gradients near $10^{-6}$ fall below the normal threshold $6.1 \times 10^{-5}$: they land in the subnormal range where relative precision degrades from $2^{-11}$ to as little as 1 bit, and values below $6 \times 10^{-8}$ flush to zero. Zeroed or coarsened gradients make weight updates vanish — training stalls.

Loss scaling multiplies the loss (hence, by linearity of backpropagation, every gradient) by $2^{k}$. We need

$$
2^{k} \times 10^{-6} \ge 2^{-14} \approx 6.1 \times 10^{-5} \iff 2^{k} \ge 61 \iff k \ge \log_2 61 \approx 5.93 .
$$

So $k = 6$, scale $= 64$. After the optimizer step direction is computed, gradients are divided by 64 in float32 master weights.

$$
\boxed{\text{loss scale } 2^{6} = 64 \text{ lifts } 10^{-6} \text{ gradients into float16's normal range}}
$$

*Key takeaway:* Mixed-precision training is applied error analysis — match the statistical dynamic range of your quantities to the exponent range of the format.

### Problem L2.3: One-pass variance is dangerous

A data pipeline computes feature variance as $\mathrm{Var} = \frac{1}{n}\sum x_i^2 - \bar{x}^2$ on the sample $x = (10^{6} + 1,\; 10^{6} + 2,\; 10^{6} + 3)$ in single precision (about 7 decimal digits). Predict the failure and give the stable alternative.

**Solution.**

Exact values: $\bar{x} = 10^{6} + 2$ and $\mathrm{Var} = \frac{(1{-}2)^2 + 0 + (3{-}2)^2}{3} = \frac{2}{3}$.

The two terms being subtracted are both $\approx 10^{12}$, while their difference is $\tfrac{2}{3} \sim 10^{0}$: the subtraction must cancel about $12$ digits, but single precision carries only $\approx 7$. The computed variance is pure rounding noise (often negative — a variance!). Amplification factor per Proof 2:

$$
\kappa \approx \frac{2 \times 10^{12}}{2/3} \approx 3 \times 10^{12}.
$$

**Stable fixes:** (i) two-pass: compute $\bar{x}$ first, then $\frac{1}{n}\sum (x_i - \bar{x})^2$ — differences are $O(1)$, no cancellation; (ii) Welford's online update $M_k = M_{k-1} + (x_k - \bar{x}_{k-1})(x_k - \bar{x}_k)$.

$$
\boxed{\text{one-pass formula loses } \approx 12 \text{ digits; use two-pass or Welford (exact answer } 2/3)}
$$

*Key takeaway:* Shift the data before squaring — the variance formula $E[x^2] - (E[x])^2$ is an identity of real numbers, not of floats.

### Problem L2.4: log1p in a log-likelihood

A logistic model's log-likelihood needs $\log(1 + e^{-t})$ for $t = 40$. Show that the naive evaluation returns 0 and quantify the correct value; then express the numerically sound evaluation for both signs of $t$.

**Solution.**

For $t = 40$: $e^{-40} \approx 4.25 \times 10^{-18} \lt u$, so $\mathrm{fl}(1 + e^{-40}) = 1$ and $\log(1) = 0$. The true value is

$$
\log(1 + e^{-40}) = e^{-40} - \tfrac{1}{2}e^{-80} + \cdots \approx 4.25 \times 10^{-18} \neq 0 .
$$

`log1p(x)` evaluates $\log(1+x)$ via a series/argument-reduction accurate for tiny $x$, returning $\approx 4.25 \times 10^{-18}$ with relative error $O(u)$. The softplus $\log(1 + e^{-t})$ should be computed as

$$
\mathrm{softplus}(-t) = \begin{cases} \mathrm{log1p}(e^{-t}), & t \ge 0, \\ -t + \mathrm{log1p}(e^{t}), & t \lt 0, \end{cases}
$$

which never exponentiates a large positive number and never adds a tiny number to 1 naively.

$$
\boxed{\log(1 + e^{-40}) \approx 4.25 \times 10^{-18}; \text{ use } \mathrm{log1p} \text{ with the sign-split formula}}
$$

*Key takeaway:* In likelihoods and losses, relative accuracy of *tiny* terms matters because gradients are taken through them.

### Problem L2.5: Relativistic kinetic energy for slow particles

Kinetic energy is $E_k = (\gamma - 1) mc^2$ with $\gamma = (1 - \beta^2)^{-1/2}$, $\beta = v/c$. For a satellite at $v = 7.7$ km/s ($\beta \approx 2.57 \times 10^{-5}$), show the naive formula cancels catastrophically in double precision and derive a stable series evaluation.

**Solution.**

$\beta^2 \approx 6.6 \times 10^{-10}$, so $\gamma = 1 + 3.3 \times 10^{-10} + O(\beta^4)$. Computing $\gamma$ first stores it as $1 + 3.3\times 10^{-10}$ with absolute rounding error up to $u \approx 10^{-16}$; the subtraction $\gamma - 1 \approx 3.3 \times 10^{-10}$ then has relative error up to

$$
\frac{u}{3.3 \times 10^{-10}} \approx 3 \times 10^{-7},
$$

i.e. about 9 of 16 digits lost (worse in single precision — total loss). Stable route: expand

$$
\gamma - 1 = \frac{1}{\sqrt{1-\beta^2}} - 1 = \tfrac{1}{2}\beta^2 + \tfrac{3}{8}\beta^4 + O(\beta^6),
$$

and evaluate $E_k = mc^2\bigl(\tfrac{1}{2}\beta^2 + \tfrac{3}{8}\beta^4\bigr)$ — no cancellation, relative error $O(u) + O(\beta^6)$; the truncation term $\beta^6 \sim 10^{-28}$ is far below $u$. (Equivalently, the exact rewrite $\gamma - 1 = \beta^2\gamma^2/(\gamma+1)$ avoids the series.)

$$
\boxed{E_k = mc^2\left(\tfrac{1}{2}\beta^2 + \tfrac{3}{8}\beta^4 + \cdots\right) \text{ — cancellation-free for } \beta \ll 1}
$$

*Key takeaway:* Physics formulas involving "relativistic minus rest" or "total minus reference" quantities should be re-derived around the reference point before coding.

### Problem L2.6: Kahan compensated summation

State Kahan's compensated summation algorithm and explain, by tracking the compensation variable through one step, why its error bound is $2u\sum\lvert x_i \rvert + O(nu^2)$ — independent of $n$ at first order — versus $(n-1)u\sum\lvert x_i \rvert$ for recursive summation. When does this matter in ML?

**Solution.**

Algorithm (running sum $s$, compensation $c$, both initialized appropriately):

1. $y = x_k - c$ (feed in the correction),
2. $t = s + y$ (large + small: the low-order bits of $y$ are lost),
3. $c = (t - s) - y$ (recovers *exactly* the lost bits, by Sterbenz-type exactness of $t - s$),
4. $s = t$.

Step 3 computes what was actually added, $(t - s)$, minus what we intended to add, $y$; since $t$ and $s$ are close, $t - s$ is exact, so $c$ captures the rounding error of step 2 with only $O(u^2)$ contamination. Each term's error is corrected at the *next* step instead of accumulating, leaving a per-term error $O(u\lvert x_k \rvert)$ rather than $O(nu\lvert x_k \rvert)$; summing gives $2u\sum \lvert x_i \rvert + O(nu^2)$ (Higham, Ch. 4).

**ML relevance:** summing $10^{9}$ float32 loss terms or gradient entries has $nu \approx 10^{9} \times 6 \times 10^{-8} \approx 60$ — the naive bound is vacuous. Kahan (or pairwise/tree reduction, as used by GPUs and `numpy.sum`) keeps the error at a few ulps.

$$
\boxed{\text{Kahan: error } \le 2u \textstyle\sum \lvert x_i \rvert + O(nu^2) \text{ — the } n\text{-fold growth is cancelled}}
$$

*Key takeaway:* When $n \times u$ approaches 1, plain accumulation is meaningless — compensation or tree reduction is mandatory.

## Level 3 — Challenge

### Problem L3.1: Error of a power

Prove: if $\hat{x} = x(1 + \varepsilon)$ with $n\lvert \varepsilon \rvert \le \tfrac{1}{10}$, then the relative error of $\hat{x}^{n}$ as an approximation to $x^{n}$ satisfies

$$
\left\lvert \frac{\hat{x}^{n} - x^{n}}{x^{n}} \right\rvert \le 1.2\, n \lvert \varepsilon \rvert .
$$

**Solution.**

Write the ratio exactly:

$$
\frac{\hat{x}^{n}}{x^{n}} = (1 + \varepsilon)^{n}, \qquad \text{so} \qquad \frac{\hat{x}^{n} - x^{n}}{x^{n}} = (1+\varepsilon)^{n} - 1 .
$$

For $\lvert \varepsilon \rvert \lt 1$, $\log(1+\varepsilon) \le \varepsilon$ for $\varepsilon \ge 0$ and $\lvert \log(1+\varepsilon) \rvert \le \frac{\lvert \varepsilon \rvert}{1 - \lvert \varepsilon \rvert}$ in general. Hence with $t = n\log(1+\varepsilon)$,

$$
\lvert t \rvert \le \frac{n \lvert \varepsilon \rvert}{1 - \lvert \varepsilon \rvert} \le \frac{0.1}{1 - 0.1} \le 0.112 .
$$

Then, using $\lvert e^{t} - 1 \rvert \le \lvert t \rvert e^{\lvert t \rvert}$,

$$
\lvert (1+\varepsilon)^{n} - 1 \rvert = \lvert e^{t} - 1 \rvert \le \lvert t \rvert e^{0.112} \le \frac{n\lvert \varepsilon \rvert}{0.9} \times 1.119 \le 1.2\, n \lvert \varepsilon \rvert .
$$

$$
\boxed{\lvert (1+\varepsilon)^{n} - 1 \rvert \le 1.2\, n\lvert \varepsilon \rvert \text{ whenever } n\lvert \varepsilon \rvert \le 0.1}
$$

*Key takeaway:* Relative errors add through products and powers — $\kappa_{x^n} = n$ holds not just to first order but with an explicit safe constant.

### Problem L3.2: Backward error of an inner product

Prove that the computed inner product $\hat{s} = \mathrm{fl}(x^{T} y)$ of vectors of length $n$ (recursive order) satisfies

$$
\hat{s} = \sum_{i=1}^{n} x_i y_i (1 + \theta_i), \qquad \lvert \theta_i \rvert \le \gamma_n := \frac{n u}{1 - n u},
$$

assuming $nu \lt 1$. Conclude backward stability.

**Solution.**

Each product incurs one rounding: $\mathrm{fl}(x_i y_i) = x_i y_i (1 + \pi_i)$, $\lvert \pi_i \rvert \le u$. Each partial-sum addition incurs one more. Term $x_i y_i$ passes through its multiplication and at most $n - 1$ additions... precisely: term $i \ge 2$ passes through $n - i + 1$ additions, term 1 through $n-1$. So

$$
\hat{s} = \sum_{i=1}^{n} x_i y_i (1 + \pi_i) \prod_{k \in K_i} (1 + \delta_k), \qquad \lvert K_i \rvert \le n - 1 .
$$

**Lemma (Higham's $\gamma_n$).** If $\lvert \delta_k \rvert \le u$ for $k = 1, \dots, m$ and $mu \lt 1$, then $\prod_{k=1}^{m} (1+\delta_k) = 1 + \theta$ with $\lvert \theta \rvert \le \gamma_m = \frac{mu}{1-mu}$.

*Proof of lemma:* $\lvert \theta \rvert \le (1+u)^{m} - 1 \le e^{mu} - 1 \le \frac{mu}{1 - mu}$, the last step since $e^{t} - 1 \le \frac{t}{1-t}$ for $0 \le t \lt 1$. $\square$

Each term has at most $m = n$ factors (one product + at most $n-1$ sums), so $\lvert \theta_i \rvert \le \gamma_n$. **Backward reading:** $\hat{s}$ is the *exact* inner product of $x$ with the perturbed vector $\tilde{y}$, $\tilde{y}_i = y_i(1+\theta_i)$, $\lvert \theta_i \rvert \le \gamma_n$ — componentwise backward stable.

$$
\boxed{\hat{s} = x^{T}\tilde{y}, \quad \lvert \tilde{y}_i - y_i \rvert \le \gamma_n \lvert y_i \rvert, \quad \gamma_n = \frac{nu}{1-nu}}
$$

*Key takeaway:* The $\gamma_n$ calculus turns long error analyses into bookkeeping — every classical matrix-algorithm bound (LU, QR, Cholesky) is built from this lemma.

### Problem L3.3: Conditioning of the exponential moving average

An exponential moving average $m_k = \beta m_{k-1} + (1-\beta) g_k$ (as in Adam, $\beta = 0.999$) is computed in float32 ($u \approx 6 \times 10^{-8}$). Analyze the propagation of a single rounding error injected at step $j$ and bound the total rounding error after $K$ steps with bounded inputs $\lvert g_k \rvert \le G$. Is the recursion stable?

**Solution.**

**Propagation.** Let a single perturbation $\eta$ enter at step $j$ ($\hat{m}_j = m_j + \eta$). The recursion is linear, so the perturbation evolves independently of the data:

$$
\hat{m}_k - m_k = \beta^{\,k-j} \eta \longrightarrow 0 \quad \text{geometrically, since } 0 \lt \beta \lt 1 .
$$

The recursion *forgets* old errors — it is exponentially stable.

**Accumulation.** Each step contributes fresh rounding error bounded by $c\,u \cdot (\beta\lvert m_{k-1} \rvert + (1-\beta)\lvert g_k \rvert) \le c\,u\,G$ (note $\lvert m_k \rvert \le G$ by induction, since $m_k$ is a convex combination). Summing the geometric decay of all injected errors:

$$
\lvert \hat{m}_K - m_K \rvert \le \sum_{j=1}^{K} \beta^{\,K-j} c\,u\,G \le \frac{c\,u\,G}{1 - \beta} = 1000\, c\, u\, G \approx 6 \times 10^{-5}\, c\, G .
$$

The steady-state rounding error is amplified by $\frac{1}{1-\beta}$ but is *bounded uniformly in $K$* — no unbounded drift.

$$
\boxed{\lvert \hat{m}_K - m_K \rvert \le \frac{c\,u\,G}{1-\beta} \text{ for all } K \text{ — stable, with amplification } (1-\beta)^{-1}}
$$

*Key takeaway:* Stability of a recurrence is governed by its error-propagation factor ($\beta \lt 1$ here); the same analysis with factor $\gt 1$ (e.g. long products, unstable recursions) predicts exponential error growth.

### Problem L3.4: An unstable recurrence for integrals

Define $I_n = \int_0^1 x^{n} e^{x-1} \, dx$. Integration by parts gives the recurrence $I_n = 1 - n I_{n-1}$ with $I_0 = 1 - e^{-1}$. Show that forward use of the recurrence is catastrophically unstable, quantify the error growth, and construct a stable backward algorithm.

**Solution.**

**Instability of forward recursion.** Let $\hat{I}_0 = I_0 + \eta$ with initial rounding error $\lvert \eta \rvert \le u$. The recurrence is linear:

$$
\hat{I}_n - I_n = -n(\hat{I}_{n-1} - I_{n-1}) \implies \hat{I}_n - I_n = (-1)^{n} n!\, \eta .
$$

The error grows like $n!$: at $n = 20$, $20!\,u \approx 2.4 \times 10^{18} \times 2.2 \times 10^{-16} \approx 540$, while the true value satisfies $0 \lt I_n \lt \int_0^1 x^n dx = \frac{1}{n+1}$ — the computed value is pure garbage (typically negative and huge).

**Stable backward recursion.** Invert the recurrence:

$$
I_{n-1} = \frac{1 - I_n}{n},
$$

which *divides* the error by $n$ at each step: starting from a crude guess at $N \gg n$ (e.g. $\hat{I}_{N} = \frac{1}{N+2}$, error $\lt \frac{1}{N+1}$), the error after stepping down to $n$ is at most

$$
\frac{1}{N+1} \cdot \frac{1}{N(N-1)\cdots(n+1)} = \frac{n!}{(N+1)!} \longrightarrow \text{negligible}.
$$

E.g. to get $I_{15}$ to full double precision, start at $N = 25$ with any $O(1)$ guess.

$$
\boxed{\text{forward error } \sim n!\,u \text{ (unstable)}; \quad \text{backward recursion } I_{n-1} = \tfrac{1 - I_n}{n} \text{ contracts errors by } \tfrac{1}{n}}
$$

*Key takeaway:* Stability belongs to the algorithm, not the formula — the same recurrence is useless in one direction and self-correcting in the other.